# 1. Load functions

In [1]:
import pandas as pd
import numpy as np
import h5py
import re
import os
from scipy.interpolate import interp1d
from scipy.signal import find_peaks


In [2]:
def bmg_to_hdf5_direct(input_excel, metadata_excel, output_hdf5):
    """
    Converts absorbance data from a 96-well plate into an HDF5 file with datasets for each well,
    and adds metadata directly based on the provided mapping template. Able to append new data to
    h5 file. 
    
    Parameters:
        input_excel: str - Path to the input Excel file from BMG plate reader.
        metadata_excel: str - Path to the Excel file containing metadata.
        output_hdf5: str - Path to the output HDF5 file.
    
    Returns:
        None
    """
    try:
        input_df = pd.read_excel(input_excel)
    except Exception as e:
        print(f"Error reading input file: {e}")
        return

    # Determine index based on presence of "ID"
    value = str(input_df.iloc[7, 0])
    print(value)
    index = 12 if "ID" in value else 11
    input_df = input_df.iloc[index:].reset_index(drop=True)

    # Generate wavelength column
    start_wl = int(re.search(r'\d+', input_df.iloc[0, 1]).group())
    second_wl = int(re.search(r'\d+', input_df.iloc[11, 1]).group())
    end_wl = int(re.search(r'\d+', input_df.iloc[-10, 1]).group()) if re.search(r'\d+', input_df.iloc[-10, 1]) else None
    resolution = second_wl-start_wl
    wavelengths = np.arange(start_wl, end_wl + 1, resolution)
    num_wavelengths = len(wavelengths)

    # Preprocess input as a NumPy array
    input_array = input_df.to_numpy()

    # Get the base name of the input file (without extension) for dataset naming
    base_name = os.path.splitext(os.path.basename(input_excel))[0]

    metadata_df = pd.read_excel(metadata_excel, header=None)
    block_size = 9
    samples_to_skip = set()

    for block_index in range(metadata_df.shape[0] // block_size):
        metadata_type = metadata_df.iloc[block_index * block_size, 0]
        if metadata_type == "Sample Type":
            for row in range(1, block_size):
                for col in range(1, metadata_df.shape[1]):
                    value = metadata_df.iloc[block_index * block_size + row, col]
                    if pd.isna(value) or str(value).strip() == '':
                        sample_name = f"{chr(64 + row)}{col:02d}"
                        samples_to_skip.add(sample_name)

    print(f"Samples skipped due to missing Sample Type: {sorted(samples_to_skip)}")

    # Open HDF5 file
    with h5py.File(output_hdf5, 'a') as hdf:
        newly_created_datasets = []  # Traack newly created datasets
        
        # Iterate through rows (A-H) and columns (1-12)
        for row_index in range(2, 10):  # Rows 2 to 9 (A-H)
            for col_index in range(1, 13):  # Columns 1 to 12
                sample_name = f"{chr(63 + row_index)}{col_index:02d}"
                if sample_name in samples_to_skip:
                    print(f"Skipping {sample_name} due to missing Sample Type metadata.")
                    continue
                absorbances = input_array[row_index::11, col_index][:num_wavelengths]
                absorbances = pd.to_numeric(absorbances, errors='coerce')
                sample_df = pd.DataFrame({'Wavelength': wavelengths, 'Absorbances': absorbances})

                # Check for NaN values using numpy
                if np.isnan(absorbances).all():
                    print(f"Skipping {sample_name} due to NaN values in absorbance.")
                    continue  # Skip to the next iteration

                # Concatenate the base name of the file with the sample name
                dataset_name = f"{base_name}_{sample_name}".replace(' ', '_').replace('-', '_')

                # Check if the dataset already exists in the HDF5 file
                if dataset_name in hdf:
                    print(f"Dataset {dataset_name} already exists. Skipping dataset and metadata.")
                    continue  # Skip this sample if the dataset already exists

                # Create the dataset in the HDF5 file
                hdf.create_dataset(dataset_name, data=sample_df.to_numpy())

                # Add the dataframe columns as attributes to the dataset
                for col in sample_df.columns:
                    hdf[dataset_name].attrs[col] = col

                print(f"Dataset added: {dataset_name}")
                newly_created_datasets.append(dataset_name)

        if newly_created_datasets:
            add_metadata_to_hdf5_helper(hdf, metadata_excel, newly_created_datasets)

    print(f"All datasets successfully created and metadata added in HDF5 file: {output_hdf5}")

def add_metadata_to_hdf5_helper(hdf, metadata_excel, newly_created_datasets):
    """
    Adds metadata attributes to datasets in an HDF5 file based on the metadata provided in an Excel file.
    
    Parameters:
        hdf: h5py.File - The HDF5 file object to which metadata will be added.
        metadata_excel: str - Path to the Excel file containing metadata.
        newly_created_datasets: list - List of newly created datasets for which metadata should be added.
    
    Returns:
        None
    """
    # Read the metadata from the Excel file
    metadata_df = pd.read_excel(metadata_excel, header=None)

    block_size = 9  # Each metadata block has 9 rows
    for block_index in range(metadata_df.shape[0] // block_size):
        metadata_type = metadata_df.iloc[block_index * block_size, 0]
        print(f"Adding metadata for: {metadata_type}")

        # Iterate over cells in the current metadata block
        for row in range(1, block_size):
            for col in range(1, metadata_df.shape[1]):
                attr_value = str(metadata_df.iloc[block_index * block_size + row, col])
                if not attr_value: continue  # Skip empty values

                # Generate sample name (e.g., B2, H12)
                sample_name = f"{chr(64 + row)}{col:02d}"
                print(f"{sample_name}: {metadata_type} - {attr_value}")

                # Match datasets in HDF5 and set metadata only for newly created datasets
                for dataset_name in newly_created_datasets:
                    if dataset_name.endswith(f"_{sample_name}") and isinstance(hdf[dataset_name], h5py.Dataset):
                        hdf[dataset_name].attrs[metadata_type] = attr_value
                        break  # Stop once the dataset is found

    print("Metadata successfully added to HDF5 file.")

# 2. Creating hdf5 file from BMG Output

## Creating Original hdf5 Dataset

In [4]:
output_hdf5 = r"HDF5 Datasets/Initialization HDF5 Files/250708 - Nickel Spectral Library.h5"

# Ni(NO3)2
# Dictionary of input Excel file paths and their corresponding metadata templates
file_pairs_no3 = {
    1: ("UV-Vis/BMG Output/Sample Data/A 30 31 34 35.xlsx", 
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - A30 31 34 35 template.xlsx"),
    2: ("UV-Vis/BMG Output/Sample Data/A 9-12.xlsx", 
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - A9-A12 template.xlsx"),
    3: ("UV-Vis/BMG Output/Sample Data/A 13-16.xlsx", 
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - A13-A16 template.xlsx"),
    4: ("UV-Vis/BMG Output/Sample Data/A 17-20.xlsx", 
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - A17-A20 template.xlsx"),
    5: ("UV-Vis/BMG Output/Sample Data/A 21 22 23 25.xlsx", 
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - A21 22 23 25 template.xlsx"),
    6: ("UV-Vis/BMG Output/Sample Data/A 25-28.xlsx", 
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - A25-A28.xlsx"),
    7: ("UV-Vis/BMG Output/Sample Data/A 36-39.xlsx", 
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - A36-A39 template.xlsx"),
    8: ("UV-Vis/BMG Output/Sample Data/A 40-43.xlsx", 
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - A40-43 template.xlsx"),
    9: ("UV-Vis/BMG Output/Sample Data/A 45-48.xlsx", 
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - A45-48 template.xlsx"),
    10: ("UV-Vis/BMG Output/Sample Data/A 49-52.xlsx", 
         "UV-Vis/UV-Vis Templates/Sample Templates/241224 - A49-52 template.xlsx"),
    11: ("UV-Vis/BMG Output/Sample Data/A 53-56.xlsx", 
         "UV-Vis/UV-Vis Templates/Sample Templates/241224 - A53-56 template.xlsx"),
    12: ("UV-Vis/BMG Output/Sample Data/A 57-60.xlsx", 
         "UV-Vis/UV-Vis Templates/Sample Templates/241224 - A57-60 template.xlsx"),
    13: ("UV-Vis/BMG Output/Sample Data/A 1-4.xlsx", 
         "UV-Vis/UV-Vis Templates/Sample Templates/241224 - A1-4 template.xlsx"),
    14: ("UV-Vis/BMG Output/Sample Data/A 5-8.xlsx", 
         "UV-Vis/UV-Vis Templates/Sample Templates/241224 - A5-8 template.xlsx"),
    15: ("UV-Vis/BMG Output/Sample Data/A 24 29 32 33.xlsx", 
         "UV-Vis/UV-Vis Templates/Sample Templates/241224 - A24 29 32 33 template.xlsx"),
}

file_pairs_br = {
    1: ("UV-Vis/BMG Output/Sample Data/B 30 31 34 35.xlsx", 
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - B30 31 34 35 template.xlsx"),
    2: ("UV-Vis/BMG Output/Sample Data/B 9-12.xlsx", 
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - B9-B12 template.xlsx"),
    3: ("UV-Vis/BMG Output/Sample Data/B 13-16.xlsx", 
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - B13-B16 template.xlsx"),
    4: ("UV-Vis/BMG Output/Sample Data/B 17-20.xlsx", 
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - B17-B20 template.xlsx"),
    5: ("UV-Vis/BMG Output/Sample Data/B 21 22 23 25.xlsx", 
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - B21 22 23 25 template.xlsx"),
    6: ("UV-Vis/BMG Output/Sample Data/B 25-28.xlsx", 
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - B25-B28.xlsx"),
    7: ("UV-Vis/BMG Output/Sample Data/B 36-39.xlsx",
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - B36-B39 template.xlsx"),
    8: ("UV-Vis/BMG Output/Sample Data/B 40-43.xlsx", 
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - B40-43 template.xlsx"),
    9: ("UV-Vis/BMG Output/Sample Data/B 45-48.xlsx",
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - B45-48 template.xlsx"),
    10: ("UV-Vis/BMG Output/Sample Data/B 49-52.xlsx",
         "UV-Vis/UV-Vis Templates/Sample Templates/241224 - B49-52 template.xlsx"),
    11: ("UV-Vis/BMG Output/Sample Data/B 53-56.xlsx",
         "UV-Vis/UV-Vis Templates/Sample Templates/241224 - B53-56 template.xlsx"),
    12: ("UV-Vis/BMG Output/Sample Data/B 57-60.xlsx",
         "UV-Vis/UV-Vis Templates/Sample Templates/241224 - B57-60 template.xlsx"),
    13: ("UV-Vis/BMG Output/Sample Data/B 1-4.xlsx", 
         "UV-Vis/UV-Vis Templates/Sample Templates/241224 - B1-4 template.xlsx"),
    14: ("UV-Vis/BMG Output/Sample Data/B 5-8.xlsx", 
         "UV-Vis/UV-Vis Templates/Sample Templates/241224 - B5-8 template.xlsx"),
    15: ("UV-Vis/BMG Output/Sample Data/B 24 29 32 33.xlsx", 
         "UV-Vis/UV-Vis Templates/Sample Templates/241224 - B24 29 32 33 template.xlsx"),
}

file_pairs_acac = {
    1: ("UV-Vis/BMG Output/Sample Data/C 1-4.xlsx",
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - C1-4 template.xlsx"),
    2: ("UV-Vis/BMG Output/Sample Data/C 6-9.xlsx",
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - C6-9 template.xlsx"),
    3: ("UV-Vis/BMG Output/Sample Data/C 10 12 15 16.xlsx", 
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - C10 12 15 16 template.xlsx"),
    4: ("UV-Vis/BMG Output/Sample Data/C 19-22.xlsx",
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - C19-22 template.xlsx"),
    5: ("UV-Vis/BMG Output/Sample Data/C 23 25 26 27.xlsx", 
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - C23 25 26 27 template.xlsx"),
    6: ("UV-Vis/BMG Output/Sample Data/C 28-31.xlsx",
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - C28-31 template.xlsx"),
    7: ("UV-Vis/BMG Output/Sample Data/C 34-37.xlsx",
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - C34-37 template.xlsx"),
    8: ("UV-Vis/BMG Output/Sample Data/C 38-41.xlsx",
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - C38-41 template.xlsx"),
    9: ("UV-Vis/BMG Output/Sample Data/C 42 43 45 46.xlsx",
        "UV-Vis/UV-Vis Templates/Sample Templates/241224 - C42 43 45 46 template.xlsx"),
    10: ("UV-Vis/BMG Output/Sample Data/C 47-50.xlsx", 
         "UV-Vis/UV-Vis Templates/Sample Templates/241224 - C47-50 template.xlsx"),
    11: ("UV-Vis/BMG Output/Sample Data/C 51-54.xlsx",
         "UV-Vis/UV-Vis Templates/Sample Templates/241224 - C51-54 template.xlsx"),
    12: ("UV-Vis/BMG Output/Sample Data/C 55-58.xlsx",
         "UV-Vis/UV-Vis Templates/Sample Templates/241224 - C55-58 template.xlsx"),
    13: ("UV-Vis/BMG Output/Sample Data/C 59-60.xlsx",
         "UV-Vis/UV-Vis Templates/Sample Templates/241224 - C59-60 template.xlsx"),
    14: ("UV-Vis/BMG Output/Sample Data/C 5 11 13 17.xlsx",
         "UV-Vis/UV-Vis Templates/Sample Templates/241224 - C5 11 13 17 template.xlsx"),
    15: ("UV-Vis/BMG Output/Sample Data/C 24 32 33.xlsx", 
         "UV-Vis/UV-Vis Templates/Sample Templates/241224 - C24 32 33 template.xlsx"),
    16: ("UV-Vis/BMG Output/Sample Data/C 14 18.xlsx", 
         "UV-Vis/UV-Vis Templates/Sample Templates/241224 - C14 18 template.xlsx"),
}

# Loop through all dictionaries and add to the same HDF5 file
for file_dict in [file_pairs_no3, file_pairs_br, file_pairs_acac]:
    for i, (input_excel, metadata_template) in file_dict.items():
        print(f"Input Excel: {input_excel}")
        print(f"Metadata Template: {metadata_template}")
        bmg_to_hdf5_direct(input_excel, metadata_template, output_hdf5)


Input Excel: UV-Vis/BMG Output/Sample Data/A 30 31 34 35.xlsx
Metadata Template: UV-Vis/UV-Vis Templates/Sample Templates/241224 - A30 31 34 35 template.xlsx
ID1: A30 31 34 35 again
Samples skipped due to missing Sample Type: ['H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07']
Dataset added: A_30_31_34_35_A01
Dataset added: A_30_31_34_35_A02
Dataset added: A_30_31_34_35_A03
Dataset added: A_30_31_34_35_A04
Dataset added: A_30_31_34_35_A05
Dataset added: A_30_31_34_35_A06
Dataset added: A_30_31_34_35_A07
Dataset added: A_30_31_34_35_A08
Dataset added: A_30_31_34_35_A09
Dataset added: A_30_31_34_35_A10
Dataset added: A_30_31_34_35_A11
Dataset added: A_30_31_34_35_A12
Dataset added: A_30_31_34_35_B01
Dataset added: A_30_31_34_35_B02
Dataset added: A_30_31_34_35_B03
Dataset added: A_30_31_34_35_B04
Dataset added: A_30_31_34_35_B05
Dataset added: A_30_31_34_35_B06
Dataset added: A_30_31_34_35_B07
Dataset added: A_30_31_34_35_B08
Dataset added: A_30_31_34_35_B09
Dataset added: A_30_31_34_35_B1

## Creating Original Control Dataset

In [5]:
# Ligand Controls

output_hdf5 = "HDF5 Datasets/Initialization HDF5 Files/250708 - Nickel Spectral Library Controls.h5"
# Dictionaryof input Excel file paths and their corresponding metadata templates for control files
file_pairs = {
    1: ("UV-Vis/BMG Output/Control Data/lig cont 1-12.xlsx",
        "UV-Vis/UV-Vis Templates/Control Templates/241224 - control templates L1-L12.xlsx"),
    2: ("UV-Vis/BMG Output/Control Data/lig cont 13-24.xlsx",
        "UV-Vis/UV-Vis Templates/Control Templates/241224 - control templates L13-L24.xlsx"),
    3: ("UV-Vis/BMG Output/Control Data/ligand controls 25-36 run 2.xlsx",
        "UV-Vis/UV-Vis Templates/Control Templates/241224 - control templates L25-L36.xlsx"),
    4: ("UV-Vis/BMG Output/Control Data/lig cont 36-43 45-48 2.xlsx",
        "UV-Vis/UV-Vis Templates/Control Templates/241224 - control templates L36-L43, L45-L48.xlsx"),
    5: ("UV-Vis/BMG Output/Control Data/lig cont 49-60 3.xlsx",
        "UV-Vis/UV-Vis Templates/Control Templates/241224 - control templates L49-L60.xlsx"),
    6: ("UV-Vis/BMG Output/Control Data/250324 - metal controls.xlsx",
        "UV-Vis/UV-Vis Templates/Control Templates/240324 - Metal Controls Template.xlsx"),
}

# Loop through the dictionary and call the function
for i, (input_excel, metadata_template) in file_pairs.items():
    print(f"Input Excel: {input_excel}")
    print(f"Metadata Template: {metadata_template}")
    bmg_to_hdf5_direct(input_excel, metadata_template, output_hdf5)

Input Excel: UV-Vis/BMG Output/Control Data/lig cont 1-12.xlsx
Metadata Template: UV-Vis/UV-Vis Templates/Control Templates/241224 - control templates L1-L12.xlsx
ID1: IZ-NB3-010 rerun
Samples skipped due to missing Sample Type: ['H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12']
Dataset added: lig_cont_1_12_A01
Dataset added: lig_cont_1_12_A02
Dataset added: lig_cont_1_12_A03
Dataset added: lig_cont_1_12_A04
Dataset added: lig_cont_1_12_A05
Dataset added: lig_cont_1_12_A06
Dataset added: lig_cont_1_12_A07
Dataset added: lig_cont_1_12_A08
Dataset added: lig_cont_1_12_A09
Dataset added: lig_cont_1_12_A10
Dataset added: lig_cont_1_12_A11
Dataset added: lig_cont_1_12_A12
Dataset added: lig_cont_1_12_B01
Dataset added: lig_cont_1_12_B02
Dataset added: lig_cont_1_12_B03
Dataset added: lig_cont_1_12_B04
Dataset added: lig_cont_1_12_B05
Dataset added: lig_cont_1_12_B06
Dataset added: lig_cont_1_12_B07
Dataset added: lig_cont_1_12_B08
Dataset added: lig_cont_1_

## Creating Repeated hdf5 Dataset

In [6]:
# Individually repeated spectra

output_hdf5 = "HDF5 Datasets/Initialization HDF5 Files/250708 - Nickel Spectral Library Repeats.h5"

# Store input Excel file paths and their corresponding metadata templates in a dictionary
file_pairs = {
    1: ("UV-Vis/BMG Output/Repeated Sample Data/Ni(NO3)2 6H2O repeat.xlsx",
        "UV-Vis/UV-Vis Templates/Sample Repeat Templates/250611 - Ni(NO3)2 6H2O repeat template.xlsx"),
    2: ("UV-Vis/BMG Output/Repeated Sample Data/NiBr2 Diglyme plate 1 repeat.xlsx",
        "UV-Vis/UV-Vis Templates/Sample Repeat Templates/250611 - NiBr2 Diglyme plate 1 repeat template.xlsx"),
    3: ("UV-Vis/BMG Output/Repeated Sample Data/NiBr2 Diglyme plate 2 repeat.xlsx",
        "UV-Vis/UV-Vis Templates/Sample Repeat Templates/250611 - NiBr2 Diglyme plate 2 repeat template.xlsx"),
    4: ("UV-Vis/BMG Output/Repeated Sample Data/Ni(acac)2 plate 1 repeat.xlsx",
        "UV-Vis/UV-Vis Templates/Sample Repeat Templates/250611 - Ni(acac)2 plate 1 repeat template.xlsx"),
    5: ("UV-Vis/BMG Output/Repeated Sample Data/Ni(acac)2 plate 2 repeat.xlsx",
        "UV-Vis/UV-Vis Templates/Sample Repeat Templates/250611 - Ni(acac)2 plate 2 repeat template.xlsx"),
}

# Loop through the dictionary and call the function
for i, (input_excel, metadata_template) in file_pairs.items():
    print(f"Input Excel: {input_excel}")
    print(f"Metadata Template: {metadata_template}")
    bmg_to_hdf5_direct(input_excel, metadata_template, output_hdf5)

Input Excel: UV-Vis/BMG Output/Repeated Sample Data/Ni(NO3)2 6H2O repeat.xlsx
Metadata Template: UV-Vis/UV-Vis Templates/Sample Repeat Templates/250611 - Ni(NO3)2 6H2O repeat template.xlsx
ID1: reru cont 2
Samples skipped due to missing Sample Type: ['A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'B06', 'B07', 'B08', 'B09', 'B10', 'B11', 'B12', 'C09', 'C10', 'C11', 'C12', 'D01', 'D02', 'D03', 'D04', 'D05', 'D06', 'D07', 'D08', 'D09', 'D10', 'D11', 'D12', 'E04', 'E05', 'E06', 'E07', 'E08', 'E09', 'E10', 'E11', 'E12', 'F07', 'F08', 'F09', 'F10', 'F11', 'F12', 'G01', 'G02', 'G03', 'G04', 'G05', 'G06', 'G07', 'G08', 'G09', 'G10', 'G11', 'G12', 'H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12']
Dataset added: Ni(NO3)2_6H2O_repeat_A01
Dataset added: Ni(NO3)2_6H2O_repeat_A02
Dataset added: Ni(NO3)2_6H2O_repeat_A03
Skipping A04 due to missing Sample Type metadata.
Skipping A05 due to missing Sample Type metadata.
Skipping A06 due to missing Samp

In [19]:
# Individually repeated ligand control spectra

output_hdf5 = "HDF5 Datasets/Initialization HDF5 Files/250708 - Nickel Spectral Library Controls Repeats.h5"

# Store input Excel file paths and their corresponding metadata templates in a dictionary
file_pairs = {
    1: ("UV-Vis/BMG Output/Repeated Sample Data/Ligand controls repeat.xlsx",
       "UV-Vis/UV-Vis Templates/Sample Repeat Templates/250612 - Ligand Control Repeats Template.xlsx")
}

# Loop through the dictionary and call the function
for i, (input_excel, metadata_template) in file_pairs.items():
    print(f"Input Excel: {input_excel}")
    print(f"Metadata Template: {metadata_template}")
    bmg_to_hdf5_direct(input_excel, metadata_template, output_hdf5)

Input Excel: UV-Vis/BMG Output/Repeated Sample Data/Ligand controls repeat.xlsx
Metadata Template: UV-Vis/UV-Vis Templates/Sample Repeat Templates/250612 - Ligand Control Repeats Template.xlsx
ID1: reru cont 2
Samples skipped due to missing Sample Type: ['A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'B08', 'B09', 'B10', 'B11', 'B12', 'C08', 'C09', 'C10', 'C11', 'C12', 'F08', 'F09', 'G06', 'G07', 'G08', 'G09', 'G10', 'G11', 'G12', 'H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12']
Dataset added: Ligand_controls_repeat_A01
Dataset added: Ligand_controls_repeat_A02
Dataset added: Ligand_controls_repeat_A03
Dataset added: Ligand_controls_repeat_A04
Dataset added: Ligand_controls_repeat_A05
Dataset added: Ligand_controls_repeat_A06
Skipping A07 due to missing Sample Type metadata.
Skipping A08 due to missing Sample Type metadata.
Skipping A09 due to missing Sample Type metadata.
Skipping A10 due to missing Sample Type metadata.
Skipping A11 due to missing Sample

# Preparing HDF5 Dataset for Modeling

In [7]:
# Remove datasets included for reproducibiltiy checks, alongside positive and negative controls
def delete_non_sample_datasets(h5_path):
    with h5py.File(h5_path, "a") as f:
        dataset_names = list(f.keys())
        for name in dataset_names:
            dset = f[name]
            sample_type = str(dset.attrs.get("Sample Type", "")).strip()
            if sample_type != "Sample":
                print(f"Deleting dataset '{name}' (Sample Type: '{sample_type}')")
                del f[name]

delete_non_sample_datasets("HDF5 Datasets/Initialization HDF5 Files/250708 - Nickel Spectral Library.h5")
delete_non_sample_datasets("HDF5 Datasets/Initialization HDF5 Files/250708 - Nickel Spectral Library Repeats.h5")

Deleting dataset 'A_13_16_H08' (Sample Type: 'Positive Control')
Deleting dataset 'A_13_16_H09' (Sample Type: 'Negative Control')
Deleting dataset 'A_13_16_H10' (Sample Type: 'Duplicate')
Deleting dataset 'A_13_16_H11' (Sample Type: 'Duplicate')
Deleting dataset 'A_13_16_H12' (Sample Type: 'Duplicate')
Deleting dataset 'A_17_20_H08' (Sample Type: 'Positive Control')
Deleting dataset 'A_17_20_H09' (Sample Type: 'Negative Control')
Deleting dataset 'A_17_20_H10' (Sample Type: 'Duplicate')
Deleting dataset 'A_17_20_H11' (Sample Type: 'Duplicate')
Deleting dataset 'A_17_20_H12' (Sample Type: 'Duplicate')
Deleting dataset 'A_1_4_H08' (Sample Type: 'Positive Control')
Deleting dataset 'A_1_4_H09' (Sample Type: 'Negative Control')
Deleting dataset 'A_1_4_H10' (Sample Type: 'Duplicate')
Deleting dataset 'A_1_4_H11' (Sample Type: 'Duplicate')
Deleting dataset 'A_1_4_H12' (Sample Type: 'Duplicate')
Deleting dataset 'A_21_22_23_25_H08' (Sample Type: 'Positive Control')
Deleting dataset 'A_21_22_2

In [ ]:
# Mapping dictionaries
metal_map = {
    "Ni(NO3)2 6H2O": "M01",
    "NiBr2 Diglyme": "M02",
    "Ni(acac)2": "M03"
}

solvent_map = {
    "DMF": "S01",
    "THF": "S02",
    "MeCN": "S03",
    "EtOH": "S04",
    "EtOAc": "S05",
    "DCE": "S06",
    "DMSO": "S07"
}

base_map = {
    "No": "B01",
    "None": "B01",
    "K2CO3": "B02",
    "DIPEA": "B03"
}

# rename and remove duplicate files
def rename_and_remove_duplicates(h5_path):
    with h5py.File(h5_path, "a") as f:
        dataset_names = list(f.keys()) 
        seen_conditions = {}

        for name in dataset_names:
            dset = f[name]

            metal_raw = str(dset.attrs.get("Metal", "")).strip()
            ligand_index = dset.attrs.get("Ligand Index", "")
            solvent_raw = str(dset.attrs.get("Solvent", "")).strip()
            base_raw = str(dset.attrs.get("Base", "")).strip()
            sample_type = str(dset.attrs.get("Sample Type", "")).strip()

            # Handle NaN ligand index → L00
            if isinstance(ligand_index, bytes):  # just in case
                ligand_index = ligand_index.decode()
            if ligand_index is None or str(ligand_index).strip() == "" or str(ligand_index).lower() == "nan":
                ligand = "L00"
            else:
                ligand_number = str(ligand_index).replace("L", "").zfill(2)
                ligand = f"L{ligand_number}"

            # Sample type suffix
            suffix_map = {
                "Positive Control": "PC",
                "Negative Control": "NC",
                "Duplicate": "D"
            }
            suffix = suffix_map.get(sample_type, "X") if sample_type != "Sample" else ""

            metal = metal_map.get(metal_raw, "M00")
            solvent = solvent_map.get(solvent_raw, "S00")
            base = base_map.get(base_raw, "B00")

            condition_key = (metal, ligand, solvent, base, suffix)
            new_name = f"{metal}_{ligand}_{solvent}_{base}"
            if suffix:
                new_name += f"_{suffix}"

            if condition_key in seen_conditions:
                print(f"Duplicate found for {condition_key}. Deleting dataset '{name}'")
                del f[name]
                continue

            # Already standardized (e.g. cell re-run): keep it instead of deleting it
            if name == new_name:
                seen_conditions[condition_key] = name
                continue

            if new_name in f:
                print(f"Dataset with name '{new_name}' already exists. Deleting '{name}'")
                del f[name]
                continue

            # Rename by copying and deleting
            data = dset[()]
            new_dset = f.create_dataset(new_name, data=data)
            for attr_key, attr_val in dset.attrs.items():
                new_dset.attrs[attr_key] = attr_val

            del f[name]
            seen_conditions[condition_key] = new_name
            print(f"Renamed '{name}' -> '{new_name}'")

# Standardize resolution to 1 nm
def standardize_resolution(h5_path):
    with h5py.File(h5_path, "a") as f:
        for name in list(f.keys()):
            dset = f[name]
            try:
                data = dset[()]
                if data.ndim != 2 or data.shape[1] != 2:
                    print(f"Skipping {name}: unexpected data shape {data.shape}")
                    continue

                wavelengths = data[:, 0]
                absorbances = data[:, 1]

                # Calculate resolution
                diffs = np.diff(wavelengths)
                resolution = np.median(diffs)

                if np.isclose(resolution, 1.0):
                    print(f"{name}: already at 1 nm resolution.")
                    continue

                # Interpolate to 1 nm
                new_wavelengths = np.arange(wavelengths.min(), wavelengths.max() + 1, 1.0)
                interp_func = interp1d(wavelengths, absorbances, kind="linear")
                new_absorbances = interp_func(new_wavelengths)

                new_data = np.column_stack((new_wavelengths, new_absorbances))

                # Overwrite the dataset
                del f[name]
                new_dset = f.create_dataset(name, data=new_data)
                for attr_key, attr_val in dset.attrs.items():
                    new_dset.attrs[attr_key] = attr_val

                print(f"{name}: interpolated from {resolution:.1f} nm to 1.0 nm.")

            except Exception as e:
                print(f"Error processing {name}: {e}")

# Crop datasets to a specific wavelength range (default: 220-700 nm)
def cut_wavelength(h5_path, min_wavelength=220, max_wavelength=700):
    print(f"\n=== Cropping datasets in '{h5_path}' to {min_wavelength}-{max_wavelength} nm window ===")

    with h5py.File(h5_path, "a") as f:
        for name in list(f.keys()):
            try:
                dset = f[name]
                data = dset[()]
                
                # Validate shape
                if data.ndim != 2 or data.shape[1] != 2:
                    print(f"Skipping {name}: unexpected data shape {data.shape}")
                    continue

                wavelengths = data[:, 0]
                absorbances = data[:, 1]

                # Apply wavelength mask
                mask = (wavelengths >= min_wavelength) & (wavelengths <= max_wavelength)
                if not np.any(mask):
                    print(f"Skipping {name}: no data in {min_wavelength}-{max_wavelength} nm range")
                    continue

                cropped_data = np.column_stack((wavelengths[mask], absorbances[mask]))

                # Overwrite the dataset
                del f[name]
                new_dset = f.create_dataset(name, data=cropped_data)
                for attr_key, attr_val in dset.attrs.items():
                    new_dset.attrs[attr_key] = attr_val

                print(f"{name}: cropped to {min_wavelength}-{max_wavelength} nm")

            except Exception as e:
                print(f"Error processing {name}: {e}")

# Standardize datasets by 
# (1) removing duplicate datasets, 
# (2) standardizing resolution to 1 nm
# (3) standardizing wavelength window to 220-700 nm

def standardize_datasets(h5_path):
    print("Renaming and removing duplicates:")
    rename_and_remove_duplicates(h5_path)
    print("Standardizing resolution (1 nm):")
    standardize_resolution(h5_path)
    print("Aligning spectral window (220 - 700 nm)")
    cut_wavelength(h5_path)

# Replace original dataset with repeated dataset
def replace_with_repeats(original_path, repeated_path, output_path):
    """
    Efficiently replaces datasets in the original HDF5 file with matching repeated datasets.
    Matching is based on 5 attributes, ignoring leading/trailing whitespace.
    A new HDF5 file is created as output.
    """
    match_attrs = ["Ligand", "Ligand Index", "Metal", "Solvent", "Base"]
    replace_count = 0
    replaced_names = []

    with h5py.File(original_path, 'r') as original_h5, \
         h5py.File(repeated_path, 'r') as repeated_h5, \
         h5py.File(output_path, 'w') as output_h5:

        # Step 1: Index original datasets by attribute tuple
        orig_key_to_name = {}
        for name, dset in original_h5.items():
            key = tuple(str(dset.attrs.get(attr, '')).strip() for attr in match_attrs)
            orig_key_to_name[key] = name
            # Copy dataset to output file
            out_dset = output_h5.create_dataset(name, data=dset[()])
            for k, v in dset.attrs.items():
                out_dset.attrs[k] = v

        # Step 2: Go through repeated datasets and replace matching originals
        for repeat_name, repeat_dset in repeated_h5.items():
            key = tuple(str(repeat_dset.attrs.get(attr, '')).strip() for attr in match_attrs)

            if key in orig_key_to_name:
                orig_name = orig_key_to_name[key]
                print(f"Replacing dataset '{orig_name}' with repeat '{repeat_name}'")

                # Replace in output file
                del output_h5[orig_name]
                out_dset = output_h5.create_dataset(orig_name, data=repeat_dset[()])
                for k, v in repeat_dset.attrs.items():
                    out_dset.attrs[k] = v

                replace_count += 1
                replaced_names.append(orig_name)

    print(f"\nCompleted. Replaced {replace_count} dataset(s). Output saved to: {output_path}")
    return replaced_names

In [9]:
standardize_datasets("HDF5 Datasets/Initialization HDF5 Files/250708 - Nickel Spectral Library.h5")
standardize_datasets("HDF5 Datasets/Initialization HDF5 Files/250708 - Nickel Spectral Library Controls.h5")
standardize_datasets("HDF5 Datasets/Initialization HDF5 Files/250708 - Nickel Spectral Library Repeats.h5")
standardize_datasets("HDF5 Datasets/Initialization HDF5 Files/250708 - Nickel Spectral Library Controls Repeats.h5")

Renaming and removing duplicates:
Renamed 'A_13_16_A01' -> 'M01_L33_S01_B01'
Renamed 'A_13_16_A02' -> 'M01_L33_S01_B02'
Renamed 'A_13_16_A03' -> 'M01_L33_S01_B03'
Renamed 'A_13_16_A04' -> 'M01_L26_S01_B01'
Renamed 'A_13_16_A05' -> 'M01_L26_S01_B02'
Renamed 'A_13_16_A06' -> 'M01_L26_S01_B03'
Renamed 'A_13_16_A07' -> 'M01_L30_S01_B01'
Renamed 'A_13_16_A08' -> 'M01_L30_S01_B02'
Renamed 'A_13_16_A09' -> 'M01_L30_S01_B03'
Renamed 'A_13_16_A10' -> 'M01_L14_S01_B01'
Renamed 'A_13_16_A11' -> 'M01_L14_S01_B02'
Renamed 'A_13_16_A12' -> 'M01_L14_S01_B03'
Renamed 'A_13_16_B01' -> 'M01_L33_S02_B01'
Renamed 'A_13_16_B02' -> 'M01_L33_S02_B02'
Renamed 'A_13_16_B03' -> 'M01_L33_S02_B03'
Renamed 'A_13_16_B04' -> 'M01_L26_S02_B01'
Renamed 'A_13_16_B05' -> 'M01_L26_S02_B02'
Renamed 'A_13_16_B06' -> 'M01_L26_S02_B03'
Renamed 'A_13_16_B07' -> 'M01_L30_S02_B01'
Renamed 'A_13_16_B08' -> 'M01_L30_S02_B02'
Renamed 'A_13_16_B09' -> 'M01_L30_S02_B03'
Renamed 'A_13_16_B10' -> 'M01_L14_S02_B01'
Renamed 'A_13_16_B11

In [10]:
replaced_names = replace_with_repeats(
    original_path="HDF5 Datasets/Initialization HDF5 Files/250708 - Nickel Spectral Library.h5",
    repeated_path="HDF5 Datasets/Initialization HDF5 Files/250708 - Nickel Spectral Library Repeats.h5",
    output_path="HDF5 Datasets/250708 - Final Nickel Spectral Library.h5"
)

replaced_names_controls = replace_with_repeats(
    original_path="HDF5 Datasets/Initialization HDF5 Files/250708 - Nickel Spectral Library Controls.h5",
    repeated_path="HDF5 Datasets/Initialization HDF5 Files/250708 - Nickel Spectral Library Controls Repeats.h5",
    output_path="HDF5 Datasets/250708 - Final Nickel Spectral Library Controls.h5"
)

Replacing dataset 'M01_L06_S02_B02' with repeat 'M01_L06_S02_B02'
Replacing dataset 'M01_L08_S03_B03' with repeat 'M01_L08_S03_B03'
Replacing dataset 'M01_L11_S06_B02' with repeat 'M01_L11_S06_B02'
Replacing dataset 'M01_L13_S03_B03' with repeat 'M01_L13_S03_B03'
Replacing dataset 'M01_L13_S05_B03' with repeat 'M01_L13_S05_B03'
Replacing dataset 'M01_L13_S06_B02' with repeat 'M01_L13_S06_B02'
Replacing dataset 'M01_L13_S06_B03' with repeat 'M01_L13_S06_B03'
Replacing dataset 'M01_L21_S02_B03' with repeat 'M01_L21_S02_B03'
Replacing dataset 'M01_L21_S03_B03' with repeat 'M01_L21_S03_B03'
Replacing dataset 'M01_L22_S06_B01' with repeat 'M01_L22_S06_B01'
Replacing dataset 'M01_L28_S02_B03' with repeat 'M01_L28_S02_B03'
Replacing dataset 'M01_L32_S05_B03' with repeat 'M01_L32_S05_B03'
Replacing dataset 'M01_L47_S03_B03' with repeat 'M01_L47_S03_B03'
Replacing dataset 'M01_L48_S01_B01' with repeat 'M01_L48_S01_B01'
Replacing dataset 'M01_L48_S01_B03' with repeat 'M01_L48_S01_B03'
Replacing 

# Extracting training dataset hdf5 files

In [ ]:
# Filter datasets based on metadata attributes
def filter_datasets(input_path, output_path,
                                dataset = None, metal=None, ligand_index=None, solvent=None, base=None, sample_type=None):
    num_datasets = 0
    with h5py.File(input_path, "r") as infile, h5py.File(output_path, "a") as outfile:
        for name in infile:
            dset = infile[name]
            attrs = dset.attrs

            # Retrieve and strip values
            metal_val = attrs.get("Metal", "").strip()
            ligand_val = attrs.get("Ligand Index", "").strip()
            solvent_val = attrs.get("Solvent", "").strip()
            base_val = attrs.get("Base", "").strip()
            sample_type_val = attrs.get("Sample Type", "").strip()

            # Check filters (skip if not in allowed list)
            if dataset and name not in dataset:
                continue
            if metal and metal_val not in metal:
                continue
            if ligand_index and ligand_val not in ligand_index:
                continue
            if solvent and solvent_val not in solvent:
                continue
            if base and base_val not in base:
                continue
            if sample_type and sample_type_val not in sample_type:
                continue

            # Copy dataset and attributes
            data = dset[()]
            new_dset = outfile.create_dataset(name, data=data)
            for attr_key, attr_val in attrs.items():
                new_dset.attrs[attr_key] = attr_val
            print(f"Copied dataset '{name}'")
            num_datasets = num_datasets + 1

    print(f"Filtered {num_datasets} datasets to: {os.path.abspath(output_path)}")

# delete specified datasets from an HDF5 file
def delete_datasets(h5_file_path, dataset_names):
    num_deleted = 0
    with h5py.File(h5_file_path, "a") as f:
        for name in dataset_names:
            if name in f:
                del f[name]
                print(f"Deleted dataset '{name}'")
                num_deleted += 1
            else:
                print(f"Dataset '{name}' not found in file")
    print(f"Deleted {num_deleted} datasets from: {os.path.abspath(h5_file_path)}")


In [ ]:
input_h5 = "HDF5 Datasets/250708 - Final Nickel Spectral Library.h5"
training_dataset_h5 = "HDF5 Datasets/250708 - Training Dataset.h5"

# Start from an empty training file so re-running this cell doesn't fail on datasets that already exist
if os.path.exists(training_dataset_h5):
    os.remove(training_dataset_h5)

# Creating initial training dataset
training_dataset_ligands = ["L05", "L48", "L56", "L12", "L32", "L33", "L04", "L02"]

filter_datasets(input_h5, 
                training_dataset_h5, 
                ligand_index = training_dataset_ligands, 
                base = ["No"], 
                sample_type = ["Sample"])

# Appending randomly selected datasets to training dataset
random_datasets = ['M03_L55_S05_B01', 'M01_L25_S07_B01', 'M02_L13_S01_B01', 'M03_L11_S03_B01', 
                   'M03_L14_S07_B01', 'M03_L14_S04_B01', 'M01_L20_S03_B01', 'M01_L19_S02_B01', 
                   'M03_L25_S05_B01', 'M01_L44_S02_B01', 'M01_L45_S05_B01', 'M03_L16_S03_B01', 
                   'M02_L45_S04_B01', 'M02_L28_S05_B01', 'M03_L39_S05_B01', 'M02_L13_S07_B01', 
                   'M02_L16_S07_B01', 'M03_L15_S04_B01', 'M01_L01_S03_B01', 'M01_L38_S01_B01', 
                   'M01_L15_S04_B01', 'M02_L09_S02_B01', 'M02_L53_S02_B01', 'M02_L13_S06_B01', 
                   'M01_L24_S03_B01', 'M01_L28_S03_B01', 'M01_L50_S07_B01', 'M03_L52_S02_B01', 
                   'M01_L26_S05_B01', 'M01_L26_S01_B01', 'M02_L17_S06_B01', 'M02_L47_S02_B01', 
                   'M02_L20_S05_B01', 'M03_L58_S05_B01', 'M01_L15_S07_B01', 'M02_L45_S03_B01', 
                   'M03_L55_S03_B01', 'M02_L47_S06_B01', 'M03_L10_S01_B01', 'M02_L10_S01_B01']


filter_datasets(input_h5, 
                training_dataset_h5, 
                dataset = random_datasets)

# Removing Ni(NO3)2 6H2O (M01) and DCE (S06) from training dataset due to insolubility (see Supporting Information)
nitrate_dce = [
    "M01_L05_S06_B01",
    "M01_L48_S06_B01", 
    "M01_L56_S06_B01",
    "M01_L12_S06_B01",
    "M01_L32_S06_B01",
    "M01_L33_S06_B01",
    "M01_L04_S06_B01",
    "M01_L02_S06_B01",
]

delete_datasets(training_dataset_h5,
                nitrate_dce)

Copied dataset 'M01_L02_S01_B01'
Copied dataset 'M01_L02_S02_B01'
Copied dataset 'M01_L02_S03_B01'
Copied dataset 'M01_L02_S04_B01'
Copied dataset 'M01_L02_S05_B01'
Copied dataset 'M01_L02_S06_B01'
Copied dataset 'M01_L02_S07_B01'
Copied dataset 'M01_L04_S01_B01'
Copied dataset 'M01_L04_S02_B01'
Copied dataset 'M01_L04_S03_B01'
Copied dataset 'M01_L04_S04_B01'
Copied dataset 'M01_L04_S05_B01'
Copied dataset 'M01_L04_S06_B01'
Copied dataset 'M01_L04_S07_B01'
Copied dataset 'M01_L05_S01_B01'
Copied dataset 'M01_L05_S02_B01'
Copied dataset 'M01_L05_S03_B01'
Copied dataset 'M01_L05_S04_B01'
Copied dataset 'M01_L05_S05_B01'
Copied dataset 'M01_L05_S06_B01'
Copied dataset 'M01_L05_S07_B01'
Copied dataset 'M01_L12_S01_B01'
Copied dataset 'M01_L12_S02_B01'
Copied dataset 'M01_L12_S03_B01'
Copied dataset 'M01_L12_S04_B01'
Copied dataset 'M01_L12_S05_B01'
Copied dataset 'M01_L12_S06_B01'
Copied dataset 'M01_L12_S07_B01'
Copied dataset 'M01_L32_S01_B01'
Copied dataset 'M01_L32_S02_B01'
Copied dat

In [13]:
def count_unique_datasets(h5_file_path):
    with h5py.File(h5_file_path, "r") as f:
        unique_datasets = list(f.keys())
        return len(unique_datasets)


train_path = "HDF5 Datasets/250708 - Training Dataset.h5"
library_path = "HDF5 Datasets/250708 - Final Nickel Spectral Library.h5"
before_filter_library_path = "HDF5 Datasets/Initialization HDF5 Files/250708 - Nickel Spectral Library.h5"
print("Number of unique datasets:", count_unique_datasets(train_path))
print("Number of unique dataset:", count_unique_datasets(before_filter_library_path))


Number of unique datasets: 200
Number of unique dataset: 3717


## UV-Vis Time Course

In [9]:
output_hdf5 = r"UV-Vis Time Course/UV-Vis Time Course.h5"

# MeCN UV-Vis
# Dictionary of input Excel file paths and their corresponding metadata templates
file_pairs_MeCN = {
    1: ("UV-Vis Time Course/M02_S03_timecourse/05192026_M02_S03_L05_L07_L33_0min.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_S03_L05_L07_L33_0min template.xlsx"),
    2: ("UV-Vis Time Course/M02_S03_timecourse/05192026_M02_S03_L05_L07_L33_5min.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_S03_L05_L07_L33_5min template.xlsx"),
    3: ("UV-Vis Time Course/M02_S03_timecourse/05192026_M02_S03_L05_L07_L33_10min.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_S03_L05_L07_L33_10min template.xlsx"),
    4: ("UV-Vis Time Course/M02_S03_timecourse/05192026_M02_S03_L05_L07_L33_15min.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_S03_L05_L07_L33_15min template.xlsx"),
    5: ("UV-Vis Time Course/M02_S03_timecourse/05192026_M02_S03_L05_L07_L33_20min.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_S03_L05_L07_L33_20min template.xlsx"),
    6: ("UV-Vis Time Course/M02_S03_timecourse/05192026_M02_S03_L05_L07_L33_25min.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_S03_L05_L07_L33_25min template.xlsx"),
    7: ("UV-Vis Time Course/M02_S03_timecourse/05192026_M02_S03_L05_L07_L33_30min.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_S03_L05_L07_L33_30min template.xlsx"),
    8: ("UV-Vis Time Course/M02_S03_timecourse/05192026_M02_S03_L05_L07_L33_cool.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_S03_L05_L07_L33_cool template.xlsx"),
}

file_pairs_tol = {
    1: ("UV-Vis Time Course/M02_tol_timecourse/05192026_M02_tol_L05_L07_L33_0min.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_tol_L05_L07_L33_0min template.xlsx"),
    2: ("UV-Vis Time Course/M02_tol_timecourse/05192026_M02_tol_L05_L07_L33_5min.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_tol_L05_L07_L33_5min template.xlsx"),
    3: ("UV-Vis Time Course/M02_tol_timecourse/05192026_M02_tol_L05_L07_L33_10min.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_tol_L05_L07_L33_10min template.xlsx"),
    4: ("UV-Vis Time Course/M02_tol_timecourse/05192026_M02_tol_L05_L07_L33_15min.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_tol_L05_L07_L33_15min template.xlsx"),
    5: ("UV-Vis Time Course/M02_tol_timecourse/05192026_M02_tol_L05_L07_L33_20min.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_tol_L05_L07_L33_20min template.xlsx"),
    6: ("UV-Vis Time Course/M02_tol_timecourse/05192026_M02_tol_L05_L07_L33_25min.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_tol_L05_L07_L33_25min template.xlsx"),
    7: ("UV-Vis Time Course/M02_tol_timecourse/05192026_M02_tol_L05_L07_L33_30min.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_tol_L05_L07_L33_30min template.xlsx"),
    8: ("UV-Vis Time Course/M02_tol_timecourse/05192026_M02_tol_L05_L07_L33_cool.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_tol_L05_L07_L33_cool template.xlsx"),
}

file_pairs_resus = {
    1: ("UV-Vis Time Course/M02_S03prestir_tol_resuspension_timecourse/05192026_M02_resus_L05_L07_L33_0min.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_resus_L05_L07_L33_0min template.xlsx"),
    2: ("UV-Vis Time Course/M02_S03prestir_tol_resuspension_timecourse/05192026_M02_resus_L05_L07_L33_5min.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_resus_L05_L07_L33_5min template.xlsx"),
    3: ("UV-Vis Time Course/M02_S03prestir_tol_resuspension_timecourse/05192026_M02_resus_L05_L07_L33_10min.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_resus_L05_L07_L33_10min template.xlsx"),
    4: ("UV-Vis Time Course/M02_S03prestir_tol_resuspension_timecourse/05192026_M02_resus_L05_L07_L33_15min.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_resus_L05_L07_L33_15min template.xlsx"),
    5: ("UV-Vis Time Course/M02_S03prestir_tol_resuspension_timecourse/05192026_M02_resus_L05_L07_L33_20min.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_resus_L05_L07_L33_20min template.xlsx"),
    6: ("UV-Vis Time Course/M02_S03prestir_tol_resuspension_timecourse/05192026_M02_resus_L05_L07_L33_25min.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_resus_L05_L07_L33_25min template.xlsx"),
    7: ("UV-Vis Time Course/M02_S03prestir_tol_resuspension_timecourse/05192026_M02_resus_L05_L07_L33_30min.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_resus_L05_L07_L33_30min template.xlsx"),
    8: ("UV-Vis Time Course/M02_S03prestir_tol_resuspension_timecourse/05192026_M02_resus_L05_L07_L33_cool.xlsx", 
        "UV-Vis Time Course/Templates/05192026_M02_resus_L05_L07_L33_cool template.xlsx"),
}

# Loop through all dictionaries and add to the same HDF5 file
for file_dict in [file_pairs_MeCN, file_pairs_tol, file_pairs_resus]:
    for i, (input_excel, metadata_template) in file_dict.items():
        print(f"Input Excel: {input_excel}")
        print(f"Metadata Template: {metadata_template}")
        bmg_to_hdf5_direct(input_excel, metadata_template, output_hdf5)


Input Excel: UV-Vis Time Course/M02_S03_timecourse/05192026_M02_S03_L05_L07_L33_0min.xlsx
Metadata Template: UV-Vis Time Course/Templates/05192026_M02_S03_L05_L07_L33_0min template.xlsx
ID1: 05192026_M02_S03_L05_L07_L33_0min
Samples skipped due to missing Sample Type: ['A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B09', 'B10', 'B11', 'B12', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07', 'C08', 'C09', 'C10', 'C11', 'C12', 'D01', 'D02', 'D03', 'D04', 'D05', 'D06', 'D07', 'D08', 'D09', 'D10', 'D11', 'D12', 'E01', 'E02', 'E03', 'E04', 'E05', 'E06', 'E07', 'E08', 'E09', 'E10', 'E11', 'E12', 'F01', 'F02', 'F03', 'F04', 'F05', 'F06', 'F07', 'F08', 'F09', 'F10', 'F11', 'F12', 'G01', 'G02', 'G03', 'G04', 'G05', 'G06', 'G07', 'G08', 'G09', 'G10', 'G11', 'G12', 'H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12']
Dataset 05192026_M02_S03_L05_L07_L33_0min_A01 already exists. Skipping dataset 

In [4]:
output_hdf5 = r"UV-Vis Time Course/UV-Vis Time Course 2.h5"

# BMG UV-Vis
# Dictionary of input Excel file paths and their corresponding metadata templates
file_pairs = {
    1: ("UV-Vis Time Course/05272026_rxn_Components_TimeCourse_BMG/05272026_M02_L05_rxn_components_0min.xlsx", 
        "UV-Vis Time Course/05272026_rxn_Components_TimeCourse_BMG/Templates/05272026_M02_L05_rxn_components_0min template.xlsx"),
    2: ("UV-Vis Time Course/05272026_rxn_Components_TimeCourse_BMG/05272026_M02_L05_rxn_components_5min.xlsx", 
        "UV-Vis Time Course/05272026_rxn_Components_TimeCourse_BMG/Templates/05272026_M02_L05_rxn_components_5min template.xlsx"),
    3: ("UV-Vis Time Course/05272026_rxn_Components_TimeCourse_BMG/05272026_M02_L05_rxn_components_10min.xlsx", 
        "UV-Vis Time Course/05272026_rxn_Components_TimeCourse_BMG/Templates/05272026_M02_L05_rxn_components_10min template.xlsx"),
    4: ("UV-Vis Time Course/05272026_rxn_Components_TimeCourse_BMG/05272026_M02_L05_rxn_components_15min.xlsx", 
        "UV-Vis Time Course/05272026_rxn_Components_TimeCourse_BMG/Templates/05272026_M02_L05_rxn_components_15min template.xlsx"),
    5: ("UV-Vis Time Course/05272026_rxn_Components_TimeCourse_BMG/05272026_M02_L05_rxn_components_20min.xlsx", 
        "UV-Vis Time Course/05272026_rxn_Components_TimeCourse_BMG/Templates/05272026_M02_L05_rxn_components_20min template.xlsx"),
    6: ("UV-Vis Time Course/05272026_rxn_Components_TimeCourse_BMG/05272026_M02_L05_rxn_components_25min.xlsx", 
        "UV-Vis Time Course/05272026_rxn_Components_TimeCourse_BMG/Templates/05272026_M02_L05_rxn_components_25min template.xlsx"),
    7: ("UV-Vis Time Course/05272026_rxn_Components_TimeCourse_BMG/05272026_M02_L05_rxn_components_30min.xlsx", 
        "UV-Vis Time Course/05272026_rxn_Components_TimeCourse_BMG/Templates/05272026_M02_L05_rxn_components_30min template.xlsx"),
    8: ("UV-Vis Time Course/05272026_rxn_Components_TimeCourse_BMG/05272026_M02_L05_rxn_components_cool.xlsx", 
        "UV-Vis Time Course/05272026_rxn_Components_TimeCourse_BMG/Templates/05272026_M02_L05_rxn_components_cool template.xlsx"),
    9: ("UV-Vis Time Course/05272026_rxn_Components_TimeCourse_BMG/05272026_M02_L05_rxn_components_cool_ALL.xlsx", 
        "UV-Vis Time Course/05272026_rxn_Components_TimeCourse_BMG/Templates/05272026_M02_L05_rxn_components_cool_ALL template.xlsx"),
}

# Loop through all dictionaries and add to the same HDF5 file
for file_dict in [file_pairs]:
    for i, (input_excel, metadata_template) in file_dict.items():
        print(f"Input Excel: {input_excel}")
        print(f"Metadata Template: {metadata_template}")
        bmg_to_hdf5_direct(input_excel, metadata_template, output_hdf5)


Input Excel: UV-Vis Time Course/05272026_rxn_Components_TimeCourse_BMG/05272026_M02_L05_rxn_components_0min.xlsx
Metadata Template: UV-Vis Time Course/05272026_rxn_Components_TimeCourse_BMG/Templates/05272026_M02_L05_rxn_components_0min template.xlsx
ID1: 05272026_M02_L05_rxn_components_0min
Samples skipped due to missing Sample Type: ['A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B09', 'B10', 'B11', 'B12', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07', 'C08', 'C09', 'C10', 'C11', 'C12', 'D02', 'D03', 'D04', 'D05', 'D06', 'D07', 'D08', 'D09', 'D10', 'D11', 'D12', 'E01', 'E02', 'E03', 'E04', 'E05', 'E06', 'E07', 'E08', 'E09', 'E10', 'E11', 'E12', 'F01', 'F02', 'F03', 'F04', 'F05', 'F06', 'F07', 'F08', 'F09', 'F10', 'F11', 'F12', 'G01', 'G02', 'G03', 'G04', 'G05', 'G06', 'G07', 'G08', 'G09', 'G10', 'G11', 'G12', 'H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12']
Dataset added: 052